[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/06-agentic-ai/06-confidence_based_escalation.ipynb)

In [1]:
# !pip install mbox openai python-dotenv

# Confidence-Based Escalation

The previous notebook ended on a score gap, `89` and `88`, one point apart, that was too close to call. Leaving that as an observation isn't enough for an agent that has to actually do something next. This notebook turns a match score, and the gap to the next-best candidate, into a real policy: proceed on its own, ask the user a clarifying question, or hand off to a human, instead of guessing every time or asking every time.

In this notebook you will:

1. Write a `decide()` policy over M|BOX's own output: score and candidate gap in, an action out
2. Run it against three real queries that land in three different buckets
3. See what each action actually looks like in practice, including the one case where the model, not M|BOX, does the talking
4. Get practical guidance on picking thresholds, because there is no universal right answer

> Note: this notebook makes real calls to the OpenAI API for the clarifying-question case. To run it, put an `OPENAI_API_KEY` in a `.env` file in this directory.

In [2]:
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
kb = pd.read_csv("datasets/kb_articles.csv")
kb[["product_id", "product_name"]]

,product_id,product_name
0,RTR-200,Wireless Router X200
1,RTR-200P,Wireless Router X200 Pro
2,CAM-410,Outdoor Security Camera 410
3,THM-050,Smart Thermostat T50
4,LGT-330,Smart Bulb Starter Kit


## 1. The policy

Three signals come out of every match: whether anything was found at all, how confident the best candidate is, and how far ahead it is of the next one. `decide()` below turns those into one of three actions. The thresholds are named arguments on purpose, they are a judgment call, not a constant, and section 4 comes back to that.

In [3]:
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

index = TableIndexer.create_index(kb, index_columns=["product_name"], tmp_dir="tmp_index_kb")
config = TableRecallConfig(
    fields=[TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                    minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
    max_results=5, min_total_match_value=0, include_field_scores=True
)

def decide(matches, high=60, low=40, ambiguous_gap=5):
    """Turn a M|BOX match result into one of AUTO_PROCEED, ASK_CLARIFYING_QUESTION, ESCALATE_TO_HUMAN."""
    if len(matches) == 0 or matches.iloc[0]["index_row"] == -1:
        return {"action": "ESCALATE_TO_HUMAN", "reason": "no candidate cleared the recall floor"}

    top = matches.iloc[0]
    if top["product_name_score"] < low:
        return {"action": "ESCALATE_TO_HUMAN", "reason": f"top score {top['product_name_score']} is below the confidence floor"}

    if len(matches) >= 2:
        runner_up = matches.iloc[1]
        gap = top["product_name_score"] - runner_up["product_name_score"]
        if runner_up["product_name_score"] >= low and gap < ambiguous_gap:
            return {
                "action": "ASK_CLARIFYING_QUESTION",
                "reason": f"top two candidates are {gap} point{'s' if gap != 1 else ''} apart",
                "candidates": [(top["product_name_candidate"], int(top["product_name_score"])),
                                (runner_up["product_name_candidate"], int(runner_up["product_name_score"]))]
            }

    if top["product_name_score"] >= high:
        return {"action": "AUTO_PROCEED", "reason": f"top score {top['product_name_score']}, no close competitor",
                "candidate": top["product_name_candidate"], "score": int(top["product_name_score"])}

    return {"action": "ASK_CLARIFYING_QUESTION", "reason": f"top score {top['product_name_score']} is plausible but not confident enough to act alone",
            "candidates": [(top["product_name_candidate"], int(top["product_name_score"]))]}

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


## 2. Three queries, three buckets

The same function, the same thresholds, three genuinely different situations.

In [4]:
queries = ["outdor camra 410", "x200", "bluetooth headphones"]

results = {}
for q in queries:
    matches = index.match(queries=pd.DataFrame({"product_name": [q]}), config=config)
    results[q] = decide(matches)
    print(f"{q!r} -> {results[q]}\n")

'outdor camra 410' -> {'action': 'AUTO_PROCEED', 'reason': 'top score 64, no close competitor', 'candidate': 'Outdoor Security Camera 410', 'score': 64}

'x200' -> {'action': 'ASK_CLARIFYING_QUESTION', 'reason': 'top two candidates are 1 point apart', 'candidates': [('Wireless Router X200', 89), ('Wireless Router X200 Pro', 88)]}

'bluetooth headphones' -> {'action': 'ESCALATE_TO_HUMAN', 'reason': 'no candidate cleared the recall floor'}



`"outdor camra 410"` clears the confidence bar with nothing else close, so it proceeds on its own. `"x200"` is genuinely ambiguous, both routers are strong matches, one point apart, so it needs a clarifying question rather than a coin flip. `"bluetooth headphones"` doesn't match anything in this catalog at all, and the policy says so honestly instead of forcing a guess.

## 3. What each action actually does

**`AUTO_PROCEED`** just answers, the way `05` did.

In [5]:
r = results["outdor camra 410"]
article = kb.loc[kb["product_name"] == r["candidate"], "article_text"].iloc[0]
print(f"Proceeding on '{r['candidate']}' (score {r['score']}):\n")
print(article)

Proceeding on 'Outdoor Security Camera 410' (score 64):

The 410 camera is weatherproofed to an IP66 rating and stores up to 30 days of footage locally on its included SD card.


**`ASK_CLARIFYING_QUESTION`** is the one case where the model, not M|BOX, should be doing the talking. M|BOX supplied the fact that matters, two specific candidates and their scores, and the model turns that into a question a person would actually want to answer, rather than a robotic score dump.

In [6]:
from openai import OpenAI
client = OpenAI()

r = results["x200"]
c1, c2 = r["candidates"]
prompt = (
    f"A customer mentioned 'x200'. Two products in our catalog are both a close match: "
    f"{c1[0]} (match score {c1[1]}) and {c2[0]} (match score {c2[1]}). "
    "Write one short, friendly clarifying question asking which one they mean."
)
response = client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": prompt}])
print(response.choices[0].message.content)

Could you please clarify if you're referring to the Wireless Router X200 or the Wireless Router X200 Pro?


**`ESCALATE_TO_HUMAN`** should not produce a sentence at all. It should produce a structured handoff a support queue can actually route, the original query and the reason it was escalated, not a paraphrased apology.

In [7]:
r = results["bluetooth headphones"]
escalation = {"query": "bluetooth headphones", "action": r["action"], "reason": r["reason"], "catalog": "kb_articles"}
escalation

{'query': 'bluetooth headphones',
 'action': 'ESCALATE_TO_HUMAN',
 'reason': 'no candidate cleared the recall floor',
 'catalog': 'kb_articles'}

## 4. Choosing thresholds is a real decision, not a formality

`high=60`, `low=40`, and `ambiguous_gap=5` were picked to make this notebook's three examples land cleanly in three different buckets, they are not values M|BOX recommends or that transfer to your data untouched. A few things worth knowing before you set your own:

- **`high` and `low` depend on how the rest of your recall config is tuned.** A stricter `IndexType` or a smaller weight on a fuzzy field will naturally push scores down across the board, thresholds have to move with them, not stay fixed.
- **`ambiguous_gap` should reflect how costly a wrong guess actually is.** Picking the wrong router might mean a customer gets a mildly wrong warranty answer, annoying but recoverable. Picking the wrong customer record before issuing a refund is a different order of consequence, and should have a much wider gap requirement, or no auto-resolution at all.
- **Log every decision along with its score and gap, not just the ones that went to a human.** The `AUTO_PROCEED` cases that were actually wrong, and the `ASK_CLARIFYING_QUESTION` cases where the user just repeated the top candidate back, are exactly the data you need to move `high`, `low`, and `ambiguous_gap` toward values that fit your real traffic instead of three cherry-picked examples.
- **Start conservative.** It costs less to ask an unnecessary clarifying question than to auto-proceed on a wrong match and have an agent act on it downstream.

## Next steps

- **`07-validating_llm_extracted_data.ipynb`** - use this same scoring to catch a model's own extraction mistakes before they become actions
- **`08-bulk_entity_resolution_for_data_agents.ipynb`** - apply the same proceed/escalate split at the scale of an entire table, not one query at a time